In [24]:
## Cell 1
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
    selection_rate,
)

INPUT_FILE = "hmda_census_cleaned.csv"
PROTECTED_ATTR = "derived_race"
RANDOM_STATE = 42

DTI_BUCKET_MAP = {
    "<20%": 15.0,
    "20%-<30%": 25.0,
    "30%-<36%": 33.0,
    "36": 36.0, "37": 37.0, "38": 38.0, "39": 39.0, "40": 40.0, "41": 41.0,
    "42": 42.0, "43": 43.0, "44": 44.0, "45": 45.0, "46": 46.0,
    "47": 47.0, "48": 48.0, "49": 49.0,
    "50%-60%": 55.0,
    ">60%": 65.0,
}

INCOME_SENTINELS = [9999]

In [26]:
## Cell 2
def load_and_prepare(path: str):
    """Loads the cleaned CSV and builds the two feature sets."""
    df = pd.read_csv(path, low_memory=False)

    df = df.dropna(subset=["approved", PROTECTED_ATTR])

    # Exclude non-demographic / mixed categories from fairness comparison
    exclude_race_values = ["Race Not Available", "Free Form Text Only", "Joint"]
    df = df[~df[PROTECTED_ATTR].isin(exclude_race_values)]

    # CAVEAT for the paper: this threshold keeps some genuinely small groups
    # (American Indian or Alaska Native, 2+ minority races, Native Hawaiian/
    # Pacific Islander are all under 1,500 rows). Fairness metrics for these
    # groups will have higher variance - report with an explicit caution
    # about sample size rather than treating them as equally precise as the
    # White/Black/Asian estimates.
    group_counts = df[PROTECTED_ATTR].value_counts()
    valid_groups = group_counts[group_counts >= 100].index
    df = df[df[PROTECTED_ATTR].isin(valid_groups)]

    df["income"] = pd.to_numeric(df["income"], errors="coerce")
    df.loc[df["income"].isin(INCOME_SENTINELS), "income"] = np.nan

    df["debt_to_income_ratio"] = df["debt_to_income_ratio"].map(DTI_BUCKET_MAP)

    baseline_features = ["income", "loan_amount", "debt_to_income_ratio"]
    baseline_features = [c for c in baseline_features if c in df.columns]

    altdata_features = baseline_features + ["rent_burden_pct", "median_household_income"]
    altdata_features = [c for c in altdata_features if c in df.columns]

    for col in altdata_features:
        missing_pct = df[col].isna().mean() * 100
        print(f"  {col}: {missing_pct:.1f}% missing, filled with median")
        df[col] = df[col].fillna(df[col].median())

    print(f"Final dataset: {len(df):,} rows after cleaning.")
    print(f"Baseline features: {baseline_features}")
    print(f"Alt-data features: {altdata_features}")
    return df, baseline_features, altdata_features

In [28]:
## Cell 3
def train_models(X_train, y_train):
    """Trains Logistic Regression, Random Forest, and XGBoost on a feature set."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    models = {
        "logistic_regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
        "xgboost": XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
    }

    fitted = {}
    for name, model in models.items():
        if name == "logistic_regression":
            model.fit(X_train_scaled, y_train)
        else:
            model.fit(X_train, y_train)
        fitted[name] = model

    return fitted, scaler

In [30]:
## Cell 4
def evaluate_accuracy(fitted_models, scaler, X_test, y_test):
    """Prints AUC for each model."""
    X_test_scaled = scaler.transform(X_test)
    results = {}
    for name, model in fitted_models.items():
        X_input = X_test_scaled if name == "logistic_regression" else X_test
        preds = model.predict_proba(X_input)[:, 1]
        auc = roc_auc_score(y_test, preds)
        results[name] = auc
        print(f"  {name}: AUC = {auc:.3f}")
    return results

In [32]:
## Cell 5
def evaluate_fairness(fitted_models, scaler, X_test, y_test, sensitive_features, label):
    """Computes disparate impact (via selection rate ratio) and equalized odds gap per model."""
    X_test_scaled = scaler.transform(X_test)
    print(f"\nFairness metrics - {label}:")
    fairness_results = {}

    for name, model in fitted_models.items():
        X_input = X_test_scaled if name == "logistic_regression" else X_test
        preds = model.predict(X_input)

        mf = MetricFrame(
            metrics=selection_rate,
            y_true=y_test,
            y_pred=preds,
            sensitive_features=sensitive_features,
        )
        rates = mf.by_group
        disparate_impact = rates.min() / rates.max()

        dp_diff = demographic_parity_difference(
            y_test, preds, sensitive_features=sensitive_features
        )
        eo_diff = equalized_odds_difference(
            y_test, preds, sensitive_features=sensitive_features
        )

        fairness_results[name] = {
            "disparate_impact_ratio": disparate_impact,
            "demographic_parity_diff": dp_diff,
            "equalized_odds_diff": eo_diff,
        }

        print(f"  {name}:")
        print(f"    Disparate impact ratio (closer to 1.0 = fairer): {disparate_impact:.3f}")
        print(f"    Demographic parity difference: {dp_diff:.3f}")
        print(f"    Equalized odds difference: {eo_diff:.3f}")

    return fairness_results

In [34]:
## Cell 6
def compare_flips(fitted_a, scaler_a, X_test_a, fitted_b, scaler_b, X_test_b,
                   sensitive_features, model_name="xgboost"):
    """Compares predictions between Model A and Model B, broken down by group."""
    X_a_input = scaler_a.transform(X_test_a) if model_name == "logistic_regression" else X_test_a
    X_b_input = scaler_b.transform(X_test_b) if model_name == "logistic_regression" else X_test_b

    preds_a = fitted_a[model_name].predict(X_a_input)
    preds_b = fitted_b[model_name].predict(X_b_input)

    flip_df = pd.DataFrame({
        "group": sensitive_features.values,
        "pred_a": preds_a,
        "pred_b": preds_b,
    })
    flip_df["newly_approved"] = (flip_df["pred_a"] == 0) & (flip_df["pred_b"] == 1)
    flip_df["newly_denied"] = (flip_df["pred_a"] == 1) & (flip_df["pred_b"] == 0)

    summary = flip_df.groupby("group")[["newly_approved", "newly_denied"]].sum()
    summary["net_change"] = summary["newly_approved"] - summary["newly_denied"]

    print(f"\n'Who pays for the fix' - flips by group ({model_name}, Model A -> Model B):")
    print(summary)
    return summary

In [36]:
## Cell 7
df, baseline_features, altdata_features = load_and_prepare(INPUT_FILE)

y = df["approved"]
sensitive = df[PROTECTED_ATTR]

  income: 1.4% missing, filled with median
  loan_amount: 0.0% missing, filled with median
  debt_to_income_ratio: 6.9% missing, filled with median
  rent_burden_pct: 0.5% missing, filled with median
  median_household_income: 0.8% missing, filled with median
Final dataset: 201,981 rows after cleaning.
Baseline features: ['income', 'loan_amount', 'debt_to_income_ratio']
Alt-data features: ['income', 'loan_amount', 'debt_to_income_ratio', 'rent_burden_pct', 'median_household_income']


In [16]:
## Cell 8
X_a = df[baseline_features]
X_a_train, X_a_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X_a, y, sensitive, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("=== Training Model A (baseline features) ===")
fitted_a, scaler_a = train_models(X_a_train, y_train)
print("\nAccuracy - Model A:")
evaluate_accuracy(fitted_a, scaler_a, X_a_test, y_test)
fairness_a = evaluate_fairness(fitted_a, scaler_a, X_a_test, y_test, sens_test, "Model A (baseline)")

=== Training Model A (baseline features) ===

Accuracy - Model A:
  logistic_regression: AUC = 0.718
  random_forest: AUC = 0.745
  xgboost: AUC = 0.800

Fairness metrics - Model A (baseline):
  logistic_regression:
    Disparate impact ratio (closer to 1.0 = fairer): 0.855
    Demographic parity difference: 0.136
    Equalized odds difference: 0.126
  random_forest:
    Disparate impact ratio (closer to 1.0 = fairer): 0.835
    Demographic parity difference: 0.139
    Equalized odds difference: 0.057
  xgboost:
    Disparate impact ratio (closer to 1.0 = fairer): 0.821
    Demographic parity difference: 0.162
    Equalized odds difference: 0.155


In [18]:
## Cell 9
X_b = df[altdata_features]
X_b_train, X_b_test, _, _, _, _ = train_test_split(
    X_b, y, sensitive, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("=== Training Model B (alt-data augmented) ===")
fitted_b, scaler_b = train_models(X_b_train, y_train)
print("\nAccuracy - Model B:")
evaluate_accuracy(fitted_b, scaler_b, X_b_test, y_test)
fairness_b = evaluate_fairness(fitted_b, scaler_b, X_b_test, y_test, sens_test, "Model B (alt-data)")

=== Training Model B (alt-data augmented) ===

Accuracy - Model B:
  logistic_regression: AUC = 0.720
  random_forest: AUC = 0.790
  xgboost: AUC = 0.806

Fairness metrics - Model B (alt-data):
  logistic_regression:
    Disparate impact ratio (closer to 1.0 = fairer): 0.869
    Demographic parity difference: 0.123
    Equalized odds difference: 0.099
  random_forest:
    Disparate impact ratio (closer to 1.0 = fairer): 0.825
    Demographic parity difference: 0.155
    Equalized odds difference: 0.119
  xgboost:
    Disparate impact ratio (closer to 1.0 = fairer): 0.822
    Demographic parity difference: 0.161
    Equalized odds difference: 0.154


In [20]:
## Cell 10
compare_flips(fitted_a, scaler_a, X_a_test, fitted_b, scaler_b, X_b_test, sens_test, model_name="xgboost")

print("\nDone. Compare fairness_a vs fairness_b above to see whether alt-data "
      "narrows or widens the fairness gap, and check the flip table to see "
      "which groups bore the cost of any change.")


'Who pays for the fix' - flips by group (xgboost, Model A -> Model B):
                                           newly_approved  newly_denied  \
group                                                                     
2 or more minority races                                1             3   
American Indian or Alaska Native                        1             3   
Asian                                                  21            32   
Black or African American                              46           132   
Native Hawaiian or Other Pacific Islander               1             1   
White                                                 296           329   

                                           net_change  
group                                                  
2 or more minority races                           -2  
American Indian or Alaska Native                   -2  
Asian                                             -11  
Black or African American                      

In [38]:
## Cell 10
compare_flips(fitted_a, scaler_a, X_a_test, fitted_b, scaler_b, X_b_test, sens_test, model_name="logistic_regression")

print("\nDone. Compare fairness_a vs fairness_b above to see whether alt-data "
      "narrows or widens the fairness gap, and check the flip table to see "
      "which groups bore the cost of any change.")


'Who pays for the fix' - flips by group (logistic_regression, Model A -> Model B):
                                           newly_approved  newly_denied  \
group                                                                     
2 or more minority races                                0             2   
American Indian or Alaska Native                        0             1   
Asian                                                  16             6   
Black or African American                              15            39   
Native Hawaiian or Other Pacific Islander               1             0   
White                                                  96            75   

                                           net_change  
group                                                  
2 or more minority races                           -2  
American Indian or Alaska Native                   -1  
Asian                                              10  
Black or African American          

In [40]:
print(sens_test.value_counts())

derived_race
White                                        40274
Black or African American                     6026
Asian                                         3671
American Indian or Alaska Native               320
2 or more minority races                       131
Native Hawaiian or Other Pacific Islander       74
Name: count, dtype: int64


In [42]:
from fairlearn.metrics import MetricFrame, selection_rate
mf = MetricFrame(metrics=selection_rate, y_true=y_test,
                 y_pred=fitted_b["logistic_regression"].predict(scaler_b.transform(X_b_test)),
                 sensitive_features=sens_test)
print(mf.by_group.sort_values())

derived_race
Native Hawaiian or Other Pacific Islander    0.810811
2 or more minority races                     0.816794
American Indian or Alaska Native             0.850000
Black or African American                    0.873880
Asian                                        0.929175
White                                        0.933555
Name: selection_rate, dtype: float64
